In [1]:
import sys
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType

spark = (SparkSession.builder
         .appName("check-python")
         .master("local[*]")
         .config("spark.pyspark.python", sys.executable)
         .config("spark.pyspark.driver.python", sys.executable)
         .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/03 09:40:25 WARN Utils: Your hostname, MacBook--Azich.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.2 instead (on interface en0)
26/03/03 09:40:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 09:40:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


---
**Что такое UDF?**

Если вы работали с SQL, то UDF для вас не новость, так как большинство традиционных баз данных RDBMS поддерживают функции, определяемые пользователем. 

Эти функции можно зарегистрировать в библиотеке базы данных и использовать их на SQL как обычные функции.

UDF в PySpark похожи на UDF в традиционных базах данных. 

В PySpark вы создаете функцию в синтаксисе Python и оборачиваете ее в PySpark SQL udf() или регистрируете ее как udf в SQL и используете ее на DataFrame и SQL соответственно.

---


**Зачем нам нужен UDF?**

UDF используются для расширения функций фреймворка, создания новых функций и повторного использования этих функций на нескольких DataFrame. Представим, что Вы хотите преобразовать каждую первую букву слова в строке имени в заглавный регистр. Встроенные функции PySpark не позволят Вам это сделать, поэтому вы можете создать UDF и использовать для этой задачи. UDF, созданные однажды, могут быть повторно использованы в нескольких DataFrame и SQL-выражениях.

---

Прежде чем создавать UDF, проверьте, нет ли аналогичной функции в Spark SQL Functions. PySpark SQL предоставляет несколько предопределенных общих функций, и с каждым выпуском добавляется все больше новых. Поэтому лучше проверить, прежде чем заново изобретать колесо.

При создании UDF необходимо очень тщательно продумать их, иначе вы столкнетесь с проблемами оптимизации и производительности, ведь UDF как правило очень дорогие (слово "очень" выделено неспроста).

---

In [2]:
# Давайте начнем с создания DF, на котором будем применять функцию.

columns = ["Seqno","Name"]

data = [("1", "john jones"),
        ("2", "tracey smith"),
        ("3", "amy sanders")]

df = spark.createDataFrame(data=data,schema=columns)
df.show(truncate=False)

+-----+------------+
|Seqno|Name        |
+-----+------------+
|1    |john jones  |
|2    |tracey smith|
|3    |amy sanders |
+-----+------------+



Создание функции Python

Первым шагом в создании UDF является создание функции Python. В приведенном ниже фрагменте создается функция convertCase(), которая принимает строковый параметр и преобразует первую букву каждого слова в заглавную.

In [3]:
def convertCase(str):
    resStr=""
    arr = str.split(" ")
    for x in arr:
       resStr= resStr + x[0:1].upper() + x[1:len(x)] + " "
    return resStr 

**Преобразование функции Python в PySpark UDF**

Теперь давайте преобразуем эту функцию convertCase() в UDF, передав ее в PySpark SQL udf(). Эта функция доступна в пакете pyspark.sql.functions. Убедитесь, что вы импортировали этот пакет перед его использованием (как в примере).

Функция PySpark SQL udf() возвращает объект класса org.apache.spark.sql.expressions.UserDefinedFunction.

In [4]:
# Converting function to UDF 

convertUDF = F.udf(lambda z: convertCase(z),StringType())

**Применение UDF к DataFrame**

Теперь вы можете использовать convertUDF() по отношению к столбцу DataFrame как обычную встроенную функцию.

In [6]:
df.select(F.col("Seqno"), \
    convertUDF(F.col("Name")).alias("Name") ) \
   .show(truncate=False)

+-----+-------------+
|Seqno|Name         |
+-----+-------------+
|1    |John Jones   |
|2    |Tracey Smith |
|3    |Amy Sanders  |
+-----+-------------+



**Использование UDF с функцией withColumn()**

Вы также можете использовать udf внутри функции DataFrame withColumn(). Чтобы объяснить это, я создам функцию upperCase(), которая преобразует входную строку в верхний регистр.

In [8]:
def upperCase(str):
    return str.upper()

upperCaseUDF = F.udf(lambda z:upperCase(z),StringType())   

df.withColumn("CAPS NAME", upperCaseUDF(F.col("Name"))) \
  .show(truncate=False)

+-----+------------+------------+
|Seqno|Name        |CAPS NAME   |
+-----+------------+------------+
|1    |john jones  |JOHN JONES  |
|2    |tracey smith|TRACEY SMITH|
|3    |amy sanders |AMY SANDERS |
+-----+------------+------------+



**Регистрация PySpark UDF и ее использование с SQL**

Чтобы использовать функцию convertCase() в PySpark SQL, вам нужно зарегистрировать ее в PySpark с помощью spark.udf.register().

In [11]:
""" Using UDF on SQL """
spark.udf.register("convertUDF", convertCase, StringType())
df.createOrReplaceTempView("NAME_TABLE")
spark.sql("select Seqno, convertUDF(Name) as Name from NAME_TABLE") \
     .show(truncate=False)

26/03/03 10:31:21 WARN SimpleFunctionRegistry: The function convertudf replaced a previously registered function.


+-----+-------------+
|Seqno|Name         |
+-----+-------------+
|1    |John Jones   |
|2    |Tracey Smith |
|3    |Amy Sanders  |
+-----+-------------+



**Итог**

* PySpark UDF - это функция, определяемая пользователем, которая используется для создания многократно используемой функции в Spark.
* Однажды созданная UDF может быть повторно использована в нескольких DataFrames и SQL (после регистрации).
* По умолчанию тип udf() - StringType.
* Вам необходимо явно обрабатывать пустые значения, иначе возникнут проблемы.